In [1]:
import pybamm
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go


v_nom = 3.63 # V

c_rates_charge = np.arange(0.1, 0.8, 0.1) 
c_rates_discharge = np.array([0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5]) 

In [2]:
parameter_values = pybamm.ParameterValues("Chen2020")
ocv_values = [parameter_values["Lower voltage cut-off [V]"]]
soc_values = [0.0]
for i in np.arange(0.01, 1, 0.01):
    soc_sim = pybamm.Simulation(
        model=pybamm.lithium_ion.SPMe(),
        parameter_values=parameter_values,
    )
    soc_sim.build(initial_soc=i)
    sol = soc_sim.step(dt=1e-6, inputs={"Current function [A]": 0.0}).last_state
    soc_values.append(i)
    ocv_values.append(sol["Surface open-circuit voltage [V]"].data[-1])
ocv_values.append(parameter_values["Upper voltage cut-off [V]"])
soc_values.append(1.0)

In [16]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=300,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{State-of-Charge}$", ticksuffix="%"
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Open-Circuit Voltage}$", ticksuffix="V"
)
fig.add_trace(
    go.Scatter(x=[soc * 100 for soc in soc_values], y=ocv_values)
)
fig.write_image("../thesis/Images/soc_estimation.pdf", "pdf")

In [4]:
n_c = []
n_c_at_vnom = []
solutions_charge = []
for i in c_rates_charge:
    experiment = pybamm.Experiment(
        [
            (
                f"Charge at {i}C until 4.2V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=0)
    solutions_charge.append(sol)
    new_n_c = 1 - sol["Current [A]"].data * (sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_c.append(new_n_c)
    n_c_at_vnom.append(new_n_c[np.argmin(np.abs(sol["Terminal voltage [V]"].data - v_nom))])
n_d = []
n_d_at_vnom = []
solutions_discharge = []
for i in c_rates_discharge:
    experiment = pybamm.Experiment(
        [
            (
                f"Discharge at {i}C until 2.5V",
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=1)
    solutions_discharge.append(sol)
    new_n_d = 1 - sol["Current [A]"].data * (-sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_d.append(new_n_d)
    n_d_at_vnom.append(new_n_d[np.argmin(np.abs(sol["Terminal voltage [V]"].data - v_nom))])
charging_efficiency = n_c_at_vnom[2]
discharging_efficiency = n_d_at_vnom[1]

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03)
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend2={"y": 0.5, "yanchor": "top"},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for i in range(len(n_c)):
    fig.add_trace(
        go.Scatter(x=solutions_charge[i]["Terminal voltage [V]"].data[1:], y=n_c[i][1:], name=f"{c_rates_charge[i]:.1f}C", legend="legend", mode="lines"), row=1, col=1 
    )
fig.add_vline(x=v_nom, line_dash="dash")
fig.add_trace(go.Scatter(x=[v_nom], y=[charging_efficiency], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=1, col=1)
fig.update_yaxes(row=1, col=1, title_text="$\\text{Charging efficiency } \\eta_c$")
for i in range(len(n_d)):
    fig.add_trace(
        go.Scatter(x=solutions_discharge[i]["Terminal voltage [V]"].data[1:], y=n_d[i][1:], name=f"-{c_rates_discharge[i]:.1f}C", legend="legend2", mode="lines"), row=2, col=1 
    )
fig.add_vline(x=v_nom, line_dash="dash")
fig.add_trace(go.Scatter(x=[v_nom], y=[discharging_efficiency], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=2, col=1)
fig.update_xaxes(row=2, col=1, title="$\\text{Terminal Voltage } V$", ticksuffix="V")
fig.update_yaxes(row=2, col=1, title_text="$\\text{Discharging efficiency } \\eta_d$")
fig.write_image("../thesis/Images/charging_efficiency.pdf", "pdf")
print(charging_efficiency)
print(discharging_efficiency)

0.9834025026791036
1.0111132120151067


In [6]:
last_charge_soc = []
for i in c_rates_charge:
    experiment = pybamm.Experiment(
        [
            (
                f"Charge at {i}C until 4.2V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPMe()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=0)
    value = sol['Surface open-circuit voltage [V]'].data[-1]
    idx = np.searchsorted(ocv_values, value, side='right')
    x1, x2 = ocv_values[idx - 1], ocv_values[idx]
    y1, y2 = soc_values[idx - 1], soc_values[idx]
    last_charge_soc.append(y1 + (value - x1) * (y2 - y1) / (x2 - x1))

last_discharge_soc = []
for i in c_rates_discharge:
    experiment = pybamm.Experiment(
        [
            (
                f"Discharge at {i}C until 2.5V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPMe()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=1)
    value = sol['Surface open-circuit voltage [V]'].data[-1]
    idx = np.searchsorted(ocv_values, value, side='right')
    x1, x2 = ocv_values[idx - 1], ocv_values[idx]
    y1, y2 = soc_values[idx - 1], soc_values[idx]
    last_discharge_soc.append(y1 + (value - x1) * (y2 - y1) / (x2 - x1))


In [7]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1)
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend2={"y": 0.5, "yanchor": "top"},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)

u_lower = np.sum(c_rates_discharge*np.array(last_discharge_soc)) / np.sum(c_rates_discharge**2)
v_lower = 0
print(u_lower, v_lower)

fig.add_trace(
    go.Scatter(x=c_rates_discharge, y=np.array(last_discharge_soc) * 100, showlegend=False, line_color="red"), row=1, col=1
)
fig.add_trace(
    go.Scatter(x=np.arange(0.0, 1.6, 0.1), y=(u_lower*np.arange(0.0, 1.6, 0.1) + v_lower) * 100, showlegend=False, mode="lines", line_color="red", line_dash="dash"), row=1, col=1
)
fig.update_yaxes(row=1, col=1, title_text="$\\text{Lower State-of-Charge Limit}$", ticksuffix="%")
fig.update_xaxes(row=1, col=1, title="$\\text{Discharge Rate}$", ticksuffix="C")


v_upper = 1
u_upper = np.sum(c_rates_charge*(np.array(last_charge_soc) - 1)) / np.sum(c_rates_charge**2)
print(u_upper, v_upper)

fig.add_trace(
    go.Scatter(x=c_rates_charge, y=np.array(last_charge_soc) * 100, showlegend=False, line_color="green"), row=2, col=1
)
fig.add_trace(
    go.Scatter(x=np.arange(0, 0.8, 0.1), y=(u_upper*np.arange(0, 0.8, 0.1) + v_upper) * 100, showlegend=False, mode="lines", line_color="green", line_dash="dash"), row=2, col=1
)
fig.update_yaxes(row=2, col=1, title_text="$\\text{Upper State-of-Charge Limit}$", ticksuffix="%")
fig.update_xaxes(row=2, col=1, title="$\\text{Charge Rate}$", ticksuffix="C")
fig.write_image("../thesis/Images/energy_limits.pdf", "pdf")

0.010179933843359025 0
-0.19530693027322138 1


In [8]:
experiment = pybamm.Experiment(
    [
        (
            "Discharge at 0.2C until 2.5V"
        )
    ],
    period="1 second"
)
model = pybamm.lithium_ion.SPMe()
sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
sol = sim.solve(initial_soc=1)
capacity = np.sum(sol["Power [W]"].data) / 3600
print(capacity)

18.782589236661295


In [9]:
clc_capacity = capacity * discharging_efficiency / (1 - last_discharge_soc[1])
print(clc_capacity)

19.05374190282412


In [15]:
import vessim as vs
import pandas as pd
from battery import CLCBattery

import nest_asyncio
nest_asyncio.apply()

class MockActor(vs.Actor):
    def __init__(self, name: str = "mock", step_size = None, p = 0):
        super().__init__(name, step_size)
        self.power = p

    def p(self, now) -> float:
        return self.power

STEP_SIZE = 1
POWER = - clc_capacity / 5

battery = CLCBattery(initial_soc=1)

environment = vs.Environment(sim_start="2022-07-03 09:00:00")
monitor = vs.Monitor(outfile="result.csv")
environment.add_microgrid(
    actors=[MockActor(p=POWER)],
    storage=battery,
    controllers=[monitor],
    step_size=STEP_SIZE,
)

environment.run(until=6 * 3600)
df = pd.read_csv("result.csv", index_col=0)
df = df.iloc[1:]
df["e_delta"] = df["e_delta"] - POWER
print(df["e_delta"].sum() / 3600) # Should be fairly close to the computed capacity

2024-06-20 13:38:30.539 | INFO     | mosaik.scenario:start:280 - Starting "Actor" as "Actor-0" ...
2024-06-20 13:38:30.539 | INFO     | mosaik.scenario:start:280 - Starting "Grid" as "Grid-0" ...
2024-06-20 13:38:30.540 | INFO     | mosaik.scenario:start:280 - Starting "Controller" as "Controller-0" ...
2024-06-20 13:38:30.541 | INFO     | mosaik.scenario:start:280 - Starting "Storage" as "Storage-0" ...
2024-06-20 13:38:30.543 | INFO     | mosaik.scenario:run:598 - Starting simulation.
100%|██████████| 21600/21600 [00:06<00:00, 3218.87steps/s]
2024-06-20 13:38:37.257 | INFO     | mosaik.scenario:run:646 - Simulation finished successfully.


18.808993586738342
